In [1]:
import pandas as pd
import os

In [2]:
data_path = r"C:\Users\prana\Desktop\RetailX Project"

In [3]:
orders = pd.read_csv(os.path.join(data_path,"olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(data_path, "olist_order_payments_dataset.csv"))
customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
category_translation = pd.read_csv(os.path.join(data_path, "product_category_name_translation.csv"))

print("All datasets loaded successfully")

All datasets loaded successfully


In [4]:
datasets = {
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "customers": customers,
    "products": products,
    "category_translation": category_translation
}

for name,df in datasets.items():
    print(f"Table: {name}")
    print(f"Rows: {df.shape[0]} , Columns: {df.shape[1]}")
    print(f"Columns: {list(df.columns)}")

Table: orders
Rows: 99441 , Columns: 8
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Table: order_items
Rows: 112650 , Columns: 7
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Table: order_payments
Rows: 103886 , Columns: 5
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Table: customers
Rows: 99441 , Columns: 5
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Table: products
Rows: 32951 , Columns: 9
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
Table: category_translation
Rows: 71 , Columns: 2
Col

In [5]:
for name, df in datasets.items():
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    if len(null_counts) > 0:
        print(f"Table: {name}")
        print(null_counts)
        print()
    else:
        print(f"Table: {name} — No nulls found\n")

Table: orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Table: order_items — No nulls found

Table: order_payments — No nulls found

Table: customers — No nulls found

Table: products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Table: category_translation — No nulls found



In [17]:
order_items[['price', 'freight_value']].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [21]:
# Check if order_items has multiple items per order
grain_check = order_items.groupby("order_id")["order_item_id"].count()
print("Max items in a single order:", grain_check.max())
print("Orders with more than 1 item:", (grain_check > 1).sum())
print("Total unique orders:", grain_check.nunique())

Max items in a single order: 21
Orders with more than 1 item: 9803
Total unique orders: 17


In [22]:
# Check how many orders have multiple payment methods
payment_check = order_payments.groupby("order_id")["payment_type"].count()
print("Orders with multiple payment rows:", (payment_check > 1).sum())
print("Max payment rows for one order:", payment_check.max())

Orders with multiple payment rows: 2961
Max payment rows for one order: 29


In [23]:
print("Total unique orders in order_items:", order_items["order_id"].nunique())
print("Total unique orders in orders table:", orders["order_id"].nunique())

Total unique orders in order_items: 98666
Total unique orders in orders table: 99441


In [24]:
# Find orders in orders table that have no matching items
orders_without_items = orders[~orders["order_id"].isin(order_items["order_id"])]
print("Orders with no items:", len(orders_without_items))
print("\nStatus breakdown of these orders:")
print(orders_without_items["order_status"].value_counts())

Orders with no items: 775

Status breakdown of these orders:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


## Key EDA Findings

1. **Grain Validation** — 9,803 orders contain multiple items (max 21 items in one order).
   This confirms grain should be one row per order item, not per order.

2. **Payment Deduplication** — 2,961 orders have multiple payment records (max 29 rows).
   This informed the ROW_NUMBER() deduplication logic in fact_sales.

3. **Missing Items** — 775 orders exist with no order items.
   All are non-delivered statuses (cancelled, unavailable etc).
   These are excluded by the WHERE order_status = 'delivered' filter in fact_sales.

4. **Product Nulls** — 610 products have no category name.
   LEFT JOIN on category translation table retains these products with NULL category.

5. **Null Columns in Orders** — delivery date columns have nulls but are excluded from our model.